In [ ]:
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv(override=True)
client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
) 


In [ ]:
messages = [
    {"role": "user", "content": "请帮我查询北京的天气情况"}
]
MODEL_NAME = "deepseek-chat"
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages
)
print(response.choices[0].message.content)

In [ ]:
response

In [ ]:
# 定义一个函数，模拟查询天气的工具
def get_weather(city):
    return {
        "城市": city,
        "天气": "晴",
        "温度": "25°C",
        "湿度": "60%",
        "风速": "5 km/h"
    }  

In [ ]:
get_weather("北京")

In [ ]:
# 将工具函数封装成符合规范的工具描述
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "查询指定城市的天气情况，一次只能查询一个城市",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "要查询的城市名称，例如：北京",
                    }
                },
                "required": ["city"]
            },
        }
    },
]

In [ ]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages,
    tools=tools,
)

In [ ]:
response

In [ ]:
response.choices[0].finish_reason

In [ ]:
response.choices[0].message.tool_calls

In [ ]:
tool_call = response.choices[0].message.tool_calls[0]
tool_call

In [ ]:
# 获取tool_call的id
tool_call.id

In [ ]:
# 获取tool_call调用的函数名称
function_name = tool_call.function.name
function_name

In [ ]:
# 获取函数调用的参数
import json
function_args = json.loads(tool_call.function.arguments)
function_args

In [ ]:
# 从参数中提取出城市名称
city = function_args.get("city")
city

In [ ]:
# 执行函数，获取结果
function_response = get_weather(city)
function_response

In [ ]:
messages

In [ ]:
response.choices[0].message.model_dump()

In [ ]:
# 追加第一次模型返回的结果消息
messages.append(response.choices[0].message.model_dump())
messages

In [ ]:
# 追加function的执行信息
messages.append({
    "role": "tool",   # 固定写法，表示这是工具调用的结果
    "tool_call_id": tool_call.id,   # 关联到具体的工具调用id
    "content": str(function_response)  # 工具执行的结果内容，必须是字符串格式
})

messages

In [ ]:
second_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages
)

In [ ]:
print(second_response.choices[0].message.content)

In [ ]:
from IPython.display import Markdown
display(Markdown(second_response.choices[0].message.content))

In [ ]:
second_response.choices[0].finish_reason